# Day 16 — Dataset Documentation and Causal Research Design

## Project

EconoCausal — Dynamic Pricing via Double Machine Learning

## Objective

Investigate the available dataset documentation and evaluate whether the AmExpert 2019 Marketing Campaign dataset supports a defensible causal research design.

## Tasks

* Inspect campaign metadata and relevant dataset documentation.
* Investigate the meaning of campaign types X and Y.
* Identify evidence of customer eligibility, treatment assignment, and campaign exposure.
* Define a potential causal research question.
* Assess the data requirements and assumptions for causal identification.
* Document unresolved questions and limitations.

## Expected Outcome

A structured assessment of the available documentation, potential research design, and evidence required before estimating causal effects using Double Machine Learning.


In [1]:
import pandas as pd
import numpy as np

# Load relevant datasets
campaigns = pd.read_csv("../data/raw/campaign_data.csv")
train = pd.read_csv("../data/raw/train.csv")

# Inspect campaign metadata
print("===== CAMPAIGN DATASET SHAPE =====")
print(campaigns.shape)

print("\n===== CAMPAIGN COLUMNS =====")
print(campaigns.columns.tolist())

print("\n===== CAMPAIGN DATA SAMPLE =====")
display(campaigns.head(10))

print("\n===== CAMPAIGN DATA TYPES =====")
display(campaigns.dtypes.astype(str).to_frame("data_type"))

# Campaign type distribution
if "campaign_type" in campaigns.columns:
    print("\n===== CAMPAIGN TYPE DISTRIBUTION =====")
    display(
        campaigns["campaign_type"]
        .value_counts(dropna=False)
        .rename_axis("campaign_type")
        .reset_index(name="campaign_count")
    )

# Inspect campaign dates, if available
date_columns = [
    col for col in campaigns.columns
    if "date" in col.lower()
]

print("\n===== CAMPAIGN DATE COLUMNS =====")
print(date_columns)

for col in date_columns:
    print(f"\n{col} values:")
    display(campaigns[col].head(10))

# Check training campaign IDs against campaign metadata
print("\n===== CAMPAIGN ID VALIDATION =====")

if "campaign_id" in train.columns and "campaign_id" in campaigns.columns:
    print(
        "Training campaign IDs missing from metadata:",
        (~train["campaign_id"].isin(campaigns["campaign_id"])).sum()
    )

print("\nCampaign metadata inspection completed.")

===== CAMPAIGN DATASET SHAPE =====
(28, 4)

===== CAMPAIGN COLUMNS =====
['campaign_id', 'campaign_type', 'start_date', 'end_date']

===== CAMPAIGN DATA SAMPLE =====


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13
5,22,X,16/09/13,18/10/13
6,18,X,10/08/13,04/10/13
7,19,Y,26/08/13,27/09/13
8,17,Y,29/07/13,30/08/13
9,16,Y,15/07/13,16/08/13



===== CAMPAIGN DATA TYPES =====


,data_type
campaign_id,int64
campaign_type,str
start_date,str
end_date,str



===== CAMPAIGN TYPE DISTRIBUTION =====


...


===== CAMPAIGN DATE COLUMNS =====
['start_date', 'end_date']

start_date values:


0    21/10/13
1    21/10/13
2    07/09/13
3    08/10/13
4    16/09/13
5    16/09/13
6    10/08/13
7    26/08/13
8    29/07/13
9    15/07/13
Name: start_date, dtype: str


end_date values:


0    20/12/13
1    22/11/13
2    16/11/13
3    15/11/13
4    18/10/13
5    18/10/13
6    04/10/13
7    27/09/13
8    30/08/13
9    16/08/13
Name: end_date, dtype: str


===== CAMPAIGN ID VALIDATION =====
Training campaign IDs missing from metadata: 0

Campaign metadata inspection completed.


In [2]:
# Day 16: Campaign Type Investigation

# Merge training records with campaign metadata
campaign_analysis = train.merge(
    campaigns[["campaign_id", "campaign_type"]],
    on="campaign_id",
    how="left",
    validate="many_to_one"
)

print("===== CAMPAIGN TYPE COUNTS =====")

display(
    campaigns["campaign_type"]
    .value_counts(dropna=False)
    .rename_axis("campaign_type")
    .reset_index(name="number_of_campaigns")
)

print("\n===== TRAINING RECORDS BY CAMPAIGN TYPE =====")

type_summary = (
    campaign_analysis
    .groupby("campaign_type", dropna=False)
    .agg(
        total_records=("redemption_status", "size"),
        unique_customers=("customer_id", "nunique"),
        unique_campaigns=("campaign_id", "nunique"),
        redemption_rate=("redemption_status", "mean")
    )
    .reset_index()
)

type_summary["redemption_rate_percent"] = (
    type_summary["redemption_rate"] * 100
).round(3)

display(type_summary)

print("\n===== CAMPAIGN-LEVEL SUMMARY =====")

campaign_summary = (
    campaign_analysis
    .groupby(["campaign_id", "campaign_type"], dropna=False)
    .agg(
        total_records=("redemption_status", "size"),
        unique_customers=("customer_id", "nunique"),
        redemption_rate=("redemption_status", "mean")
    )
    .reset_index()
)

campaign_summary["redemption_rate_percent"] = (
    campaign_summary["redemption_rate"] * 100
).round(3)

display(campaign_summary.sort_values("campaign_id"))

print(
    "\nInterpretation: These summaries describe observed redemption "
    "patterns. They do not establish why campaign types differ or "
    "whether either type caused a change in redemption."
)

===== CAMPAIGN TYPE COUNTS =====


,campaign_type,number_of_campaigns
0,Y,22
1,X,6



===== TRAINING RECORDS BY CAMPAIGN TYPE =====


,campaign_type,total_records,unique_customers,unique_campaigns,redemption_rate,redemption_rate_percent
0,X,57684,1379,4,0.011026,1.103
1,Y,20685,800,14,0.004496,0.450



===== CAMPAIGN-LEVEL SUMMARY =====


,campaign_id,campaign_type,total_records,unique_customers,redemption_rate,redemption_rate_percent
0,1,Y,143,13,0.006993,0.699
1,2,Y,768,48,0.006510,0.651
2,3,Y,408,12,0.004902,0.490
3,4,Y,972,81,0.007202,0.720
4,5,Y,1827,166,0.003831,0.383
5,6,Y,65,65,0.015385,1.538
6,7,Y,1584,198,0.002525,0.253
7,8,X,22555,1076,0.008823,0.882
8,9,Y,3168,176,0.007260,0.726
9,10,Y,1723,123,0.004063,0.406



Interpretation: These summaries describe observed redemption patterns. They do not establish why campaign types differ or whether either type caused a change in redemption.


## Step 4 — Dataset Documentation Review

### Purpose

Investigate whether the available dataset metadata establishes the meaning of campaign types X and Y and provides evidence of customer-level treatment assignment.

### Information Available

The campaign metadata contains campaign identifiers, campaign types, and campaign dates where available. The training dataset links customer and coupon records to campaign identifiers and includes the observed `redemption_status` outcome.

### Questions Requiring Verification

1. What do campaign types X and Y represent according to the original dataset documentation?
2. Were customers randomly assigned to campaigns, selected through eligibility rules, or targeted through another process?
3. Are customer-level campaign assignment or delivery records available?
4. Does the dataset record the actual discount or intervention offered to each customer?
5. What information establishes whether customers were exposed to a campaign before the observed outcome?

### Interpretation

Campaign metadata and observed redemption patterns provide descriptive information. They do not independently establish treatment assignment, randomization, or actual customer exposure.

The meaning of campaign types X and Y must be verified from the original dataset documentation or other reliable evidence before they can be used to define a causal treatment contrast.

### Preliminary Conclusion

The available metadata supports campaign-level descriptive analysis, but the treatment interpretation and assignment mechanism require further verification.


In [3]:
# Day 16: Dataset Documentation and Causal Research Design Assessment

assessment = pd.DataFrame([
    {
        "assessment_area": "Campaign metadata",
        "finding": "Campaign identifiers, types, and available dates were inspected",
        "status": "Inspected"
    },
    {
        "assessment_area": "Campaign type interpretation",
        "finding": "The meaning of campaign types X and Y requires verification from reliable documentation",
        "status": "Unverified"
    },
    {
        "assessment_area": "Observed redemption",
        "finding": "Redemption rates were compared descriptively across campaign types",
        "status": "Descriptive analysis"
    },
    {
        "assessment_area": "Customer assignment",
        "finding": "The campaign metadata and descriptive comparison do not establish the assignment mechanism",
        "status": "Unresolved"
    },
    {
        "assessment_area": "Customer eligibility",
        "finding": "Eligibility rules have not been established by the current investigation",
        "status": "Unresolved"
    },
    {
        "assessment_area": "Actual campaign exposure",
        "finding": "Actual customer-level delivery or exposure requires additional evidence",
        "status": "Unverified"
    },
    {
        "assessment_area": "Treatment definition",
        "finding": "A defensible treatment contrast has not yet been established",
        "status": "Unresolved"
    },
    {
        "assessment_area": "Causal research design",
        "finding": "The available evidence is insufficient to justify a causal effect estimate at this stage",
        "status": "Further investigation required"
    }
])

display(assessment)

# Save the assessment
assessment.to_csv(
    "../reports/day16_causal_research_design_assessment.csv",
    index=False
)

print("Day 16 assessment saved successfully.")

,assessment_area,finding,status
0,Campaign metadata,"Campaign identifiers, types, and available dat...",Inspected
1,Campaign type interpretation,The meaning of campaign types X and Y requires...,Unverified
2,Observed redemption,Redemption rates were compared descriptively a...,Descriptive analysis
3,Customer assignment,The campaign metadata and descriptive comparis...,Unresolved
4,Customer eligibility,Eligibility rules have not been established by...,Unresolved
5,Actual campaign exposure,Actual customer-level delivery or exposure req...,Unverified
6,Treatment definition,A defensible treatment contrast has not yet be...,Unresolved
7,Causal research design,The available evidence is insufficient to just...,Further investigation required


Day 16 assessment saved successfully.
